In [ ]:
import os
os.environ.pop("TRANSFORMERS_CACHE", None)        # 取消旧变量
os.environ["HF_HOME"] = "original-layout/.cache/huggingface"

# 指定只让当前进程看到第 0 张 GPU
#os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2" 

#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
#os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
work_dir = "original-layout/Projects/Jupyter/deepseek-prover-V2/runs/FormalMATH_eval_results/s2-20260103-235415"

## 结果汇总

In [ ]:
import json, jsonlines
from pathlib import Path

import json, jsonlines, math
from pathlib import Path
import time
import re



def summarize_from_summary(summary_file: str):
    """
    给定一个 summary.jsonl 文件，统计：
      - problem_pass_rate
      - answer_pass_rate
      - first_pass_rate
      - first_pass_over_passed
    统计：
      - all_time:  min / max / avg
      - pass_time: min / max / avg（仅 pass==True 的样本）
    
    并打印出来，同时返回一个 dict。
    """
    summary_path = Path(summary_file)
    if not summary_path.exists():
        print(f"[summarize] {summary_file} 不存在，无法统计")
        return {}

    total_problems = 0
    passed_problems = 0
    total_answers = 0
    passed_answers = 0
    first_pass_problems = 0
    
    # ---- time_sec（全部样本）----
    all_cnt = 0
    all_sum = 0.0
    all_min = None
    all_max = None
    
    # ---- time_sec（仅通过样本）----
    pass_cnt = 0
    pass_sum = 0.0
    pass_min = None
    pass_max = None
    
    # 记录总样本数的观察值：避免最后只打印到最后一行的 n_s
    last_n_s = None
    
    
    
    def _accum(t: float, is_pass: bool):
        nonlocal all_cnt, all_sum, all_min, all_max
        nonlocal pass_cnt, pass_sum, pass_min, pass_max

        # all
        all_cnt += 1
        all_sum += t
        all_min = t if all_min is None else min(all_min, t)
        all_max = t if all_max is None else max(all_max, t)

        # pass-only
        if is_pass:
            pass_cnt += 1
            pass_sum += t
            pass_min = t if pass_min is None else min(pass_min, t)
            pass_max = t if pass_max is None else max(pass_max, t)
    
    

    with open(summary_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue

            total_problems += 1

            if row.get("pass", False):
                passed_problems += 1
                
            if row.get("first_pass", False):
                first_pass_problems += 1

            # 每题生成多少个回答
            n_s = row.get("n_samples")
            if n_s is None:
                # 兜底：没有 n_samples 就用 attempts 的长度
                n_s = len(row.get("attempts", []))
            try:
                n_s = int(n_s)
            except Exception:
                n_s = 0
            last_n_s = n_s

            successes = row.get("successes", 0)
            try:
                successes = int(successes)
            except Exception:
                successes = 0

            total_answers += n_s
            passed_answers += successes
            

            # -------- time_sec：优先从 attempts[*].time_sec 读 --------
            attempts = row.get("attempts")
            if isinstance(attempts, list) and attempts:
                for att in attempts:
                    t = att.get("time_sec", None)
                    if t is None:
                        continue
                    try:
                        t = float(t)
                    except Exception:
                        continue
                    _accum(t, is_pass=(att.get("pass") is True))
            else:
                # 旧结构兜底：row.time_sec
                t = row.get("time_sec", None)
                if t is not None:
                    try:
                        t = float(t)
                        _accum(t, is_pass=row_pass)
                    except Exception:
                        pass
    
    
    
    print("\n[SUMMARY FROM FILE]")
    print(f"  samples (per problem): {n_s}")
    
    # 1) 题通过率：有题过就算过
    problem_pass_rate = 100.0 * passed_problems / total_problems if total_problems else 0.0
    print(f"  problem_pass_rate: total = {total_problems}  passed = {passed_problems}  (problem_pass_rate={problem_pass_rate:.2f}%)")

    # 2) 回答通过率：所有通过的回答 / 所有生成的回答
    answer_pass_rate = 100.0 * passed_answers / total_answers if total_answers else 0.0
    print(f"  answer_pass_rate : total = {total_answers}  passed = {passed_answers}  (answer_pass_rate={answer_pass_rate:.2f}%)")
    
    # 3) 第一通过率：第一个回答就通过的题目 / 所有题目
    first_pass_rate   = 100.0 * first_pass_problems / total_problems if total_problems else 0.0
    print(f"  first_pass_rate: first_pass = {first_pass_problems}  (first_pass_rate = {first_pass_rate:.2f}%)")
    
    # 4) 第一通过占比：第一个回答就通过的题目 / 总通过的题目数
    first_pass_over_passed = (
        100.0 * first_pass_problems / passed_problems if passed_problems else 0.0
    )
    print(f"  first_pass_over_passed: {first_pass_over_passed:.2f}%")        
    
    
    
    # ---- time 输出 ----
    all_avg = (all_sum / all_cnt) if all_cnt else None
    pass_avg = (pass_sum / pass_cnt) if pass_cnt else None

    if all_cnt:
        print(f"  time_sec (ALL):  count = {all_cnt}  min = {all_min:.4f}  max = {all_max:.4f}  avg = {all_avg:.4f}")
    else:
        print("  time_sec (ALL):  没有找到可用数据")

    if pass_cnt:
        print(f"  time_sec (PASS): count = {pass_cnt}  min = {pass_min:.4f}  max = {pass_max:.4f}  avg = {pass_avg:.4f}")
    else:
        print("  time_sec (PASS): 没有找到可用数据（或没有 pass==True 的样本）")

    
    

    return dict(
        total_problems = total_problems,
        passed_problems = passed_problems,
        first_pass_problems = first_pass_problems,
        total_answers = total_answers,
        passed_answers = passed_answers,
        problem_pass_rate = problem_pass_rate,
        answer_pass_rate = answer_pass_rate,
        first_pass_rate = first_pass_rate,
        first_pass_over_passed = first_pass_over_passed,
        all_time_count=all_cnt,
        all_time_min=all_min,
        all_time_max=all_max,
        all_time_avg=all_avg,
        pass_time_count=pass_cnt,
        pass_time_min=pass_min,
        pass_time_max=pass_max,
        pass_time_avg=pass_avg,
    )

In [ ]:


sum_path = work_dir + "/summary.jsonl"

In [ ]:
summarize_from_summary(sum_path)

## 生成data.jsonl
### 将summary.jsonl所有中pass的回答抽出（同一题目不同回答算不同的样本）保存为data.jsonl

In [ ]:
import re, hashlib
from pathlib import Path
import jsonlines

# --- 把 lean 证明体切成 theorem / proof 两段（不含 header）---
_TOP_DECL_RE = re.compile(r'\b(theorem|lemma|example)\b', re.M)

def split_theorem_and_proof(lean_text: str):
    """
    输入：不包含 header 的 Lean 证明源码（通常是一个定理的完整块）
    输出：(theorem_part, proof_part)
    - theorem_part：从 (theorem|lemma|example) 开始，直到 ':= by'（若无 by，则到 ':='）为止，并保留终止标记
    - proof_part  ：紧随其后的证明体；若无法解析则空串
    
    保留缩进/空白信息版：
    - 不使用 strip/lstrip/rstrip（避免丢失 proof 的行首缩进）
    - 仅统一换行符为 '\n'
    - theorem_part: 从声明开始到 ':= by'（含 by）或 ':='（含 :=）
    - proof_part  : 紧随其后的内容（原样保留，包括换行与缩进）
    """
    if lean_text is None:
        return "", ""

    text = lean_text.replace("\r\n", "\n").replace("\r", "\n")

    # 对齐到第一个顶层声明起点（不会破坏后续缩进）
    mdecl = _TOP_DECL_RE.search(text)
    if mdecl:
        text = text[mdecl.start():]  # 不 lstrip

    # 优先 ':= by'
    m_by = re.search(r':=\s*by\b', text, flags=re.M)
    if m_by:
        t_end = m_by.end()  # 包含 'by'
        theorem_part = text[:t_end]      # 不 rstrip
        proof_part   = text[t_end:]      # 不 lstrip
        return theorem_part, proof_part

    # 其次 ':='
    m_set = re.search(r':=', text, flags=re.M)
    if m_set:
        t_end = m_set.end()  # 包含 ':='
        theorem_part = text[:t_end]
        proof_part   = text[t_end:]
        return theorem_part, proof_part

    return text, ""


def extract_pass_to_jsonl(
    summary_path: str,
    out_path: str = "data.jsonl",
    keep_fields=None,          # list[str] | None
    dedup: bool = False,
) -> int:
    """
    从 summary.jsonl 提取所有 pass==True 的回答(=attempt)，写到 out_path(JSONL)。
    
    dedup：
    是否去除重复回答
    
    去重规则：
    header 相同 + lean_block（定理+证明）完全相同 的样本视为重复

    输出的“最核心字段”（无论如何都会写出）：
      id, source_id, sample_index, name, split, header, theorem, proof, lean_block

    keep_fields 规则：
    - keep_fields 为 None 或空列表：只写最核心字段
    - keep_fields 非空：除了最核心字段，还会额外拷贝 keep_fields 指定的字段
      * 字段查找顺序：attempt 优先，其次 row
      * 也支持显式前缀： "row.xxx" 或 "attempt.xxx"
      * 若字段在对应/全部原数据中不存在：直接报错 “xx字段在原数据中不存在”
      
    保留缩进/空白信息版：
    - 不对 header / lean_block 做 strip/lstrip/rstrip
    - split_theorem_and_proof 使用 keep_ws 版本
    - dedup 依据：header + lean_block 的“原样字节串”hash（不做 strip）
    """
    summary_path = Path(summary_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if keep_fields is None:
        keep_fields = []
    if not isinstance(keep_fields, (list, tuple)):
        raise TypeError("keep_fields 必须是 list/tuple 或 None")
    keep_fields = [str(x) for x in keep_fields]

    CORE_KEYS = {"id", "source_id", "sample_index", "name", "split", "header", "theorem", "proof", "lean_block"}

    def _resolve_file(p: str) -> Path:
        pp = Path(p)
        if pp.exists():
            return pp
        return summary_path.parent / pp  # 兼容相对路径

    def _get_field(field: str, row: dict, att: dict | None):
        if field.startswith("row."):
            k = field[4:]
            if k in row:
                return row[k]
            raise KeyError(f"{field}字段在原数据中不存在")
        if field.startswith("attempt."):
            k = field[8:]
            if att is not None and k in att:
                return att[k]
            raise KeyError(f"{field}字段在原数据中不存在")

        if att is not None and field in att:
            return att[field]
        if field in row:
            return row[field]
        raise KeyError(f"{field}字段在原数据中不存在")

    written = 0
    seen = set()

    with jsonlines.open(summary_path, "r") as reader, jsonlines.open(out_path, "w") as writer:
        for row in reader:
            header = row.get("header") or ""   # 原样保留
            rid = str(row.get("id") or "")
            name = row.get("name")
            split = row.get("split")

            attempts = row.get("attempts")

            # -------- 新结构：attempts 列表 --------
            if isinstance(attempts, list) and attempts:
                for att in attempts:
                    if att.get("pass") is not True:
                        continue

                    idx = int(att.get("index", 1))
                    new_id = f"{rid}_s{idx:02d}" if rid else f"s{idx:02d}"

                    lean_block = att.get("lean_block")
                    if not (isinstance(lean_block, str) and lean_block != ""):
                        lp = att.get("lean_path")
                        if lp:
                            p = _resolve_file(lp)
                            if p.exists():
                                file_src = p.read_text(encoding="utf-8")
                                file_src = file_src.replace("\r\n", "\n").replace("\r", "\n")
                                mdecl = _TOP_DECL_RE.search(file_src)
                                lean_block = file_src[mdecl.start():] if mdecl else file_src  # 不 strip

                    if not (isinstance(lean_block, str) and lean_block != ""):
                        continue

                    theorem_part, proof_part = split_theorem_and_proof(lean_block)

                    if dedup:
                        # 原样去重：header + "\n<<SEP>>\n" + lean_block
                        full_key = header + "\n<<SEP>>\n" + lean_block
                        h = hashlib.sha256(full_key.encode("utf-8")).hexdigest()
                        if h in seen:
                            continue
                        seen.add(h)

                    rec = {
                        "id": new_id,
                        "source_id": rid,
                        "sample_index": idx,
                        "name": name,
                        "split": split,
                        "header": header,
                        "theorem": theorem_part,
                        "proof": proof_part,
                        "lean_block": lean_block,
                    }

                    for f in keep_fields:
                        if f in CORE_KEYS:
                            continue
                        rec[f] = _get_field(f, row, att)

                    writer.write(rec)
                    written += 1

            # -------- 旧结构：row 自己有 pass/lean_block --------
            else:
                if row.get("pass") is not True:
                    continue
                lean_block = row.get("lean_block")
                if not (isinstance(lean_block, str) and lean_block != ""):
                    continue

                idx = 1
                new_id = f"{rid}_s01" if rid else "s01"

                theorem_part, proof_part = split_theorem_and_proof(lean_block)

                if dedup:
                    full_key = header + "\n<<SEP>>\n" + lean_block
                    h = hashlib.sha256(full_key.encode("utf-8")).hexdigest()
                    if h in seen:
                        continue
                    seen.add(h)

                rec = {
                    "id": new_id,
                    "source_id": rid,
                    "sample_index": idx,
                    "name": name,
                    "split": split,
                    "header": header,
                    "theorem": theorem_part,
                    "proof": proof_part,
                    "lean_block": lean_block,   
                }

                for f in keep_fields:
                    if f in CORE_KEYS:
                        continue
                    rec[f] = _get_field(f, row, None)

                writer.write(rec)
                written += 1

    print(f"[DATA] wrote {written} passed samples to {out_path}")
    return written

In [ ]:

data_path = work_dir + "/data.jsonl"

n = extract_pass_to_jsonl(
    sum_path,
    data_path,
    keep_fields=["domain", "source", "formal_statement", "attempt.time_sec", "attempt.lean_path", "attempt.log_path"]
)


print(n)

## sketch.jsonl的制作

### sketchify_with_fills函数

In [ ]:
def sketchify_with_fills(lean_text: str, marker: str = "<proof>"):
    """
    输入：lean_text (str)
    输出：(sketch_text, fill)

    sketch_text:
      - 按规则把部分内容替换成 marker（默认 <proof>）

    fill:
      - list[str]，按出现顺序保存“被替换掉的原始文本片段”
      - 特别地：被替换片段里会保留 `by`（原样），保证按顺序回填可还原原文本

    回填还原方式（保证成立）：
        recon = sketch_text
        for f in fill:
            recon = recon.replace(marker, f, 1)
      recon == 原文本（但会先过滤掉行内容恰为 "theorem:" / "proof:" 的行）
    """
    import re
    from typing import List, Tuple

    if lean_text is None:
        return "", []

    # 与你原程序一致的行内 by 匹配（不跨行）
    _BY_INLINE_RE = re.compile(r'\bby\s+(?!$)[^)\}\u27E9\u3009\u232A\n,;:]+')

    def _indent(s: str) -> int:
        return len(s) - len(s.lstrip(" "))

    # 去掉你原来就会跳过的标签行
    raw_lines = [ln for ln in lean_text.splitlines()
                 if ln.strip() not in {"theorem:", "proof:"}]

    out: List[str] = []
    fill: List[str] = []
    i, n = 0, len(raw_lines)

    while i < n:
        line = raw_lines[i]
        base = _indent(line)

        # 只把 '--' 之后当作行注释（Lean 单行注释）
        cidx = line.find("--")
        if cidx != -1:
            code = line[:cidx]
            comment = line[cidx:]  # 含 "--" 原样保留
        else:
            code = line
            comment = ""

        # ---------- (A) 单行 have/suffices:  `have ... := rhs` 且不是 `:= by` ----------
        # 把 rhs 整段替换为 <proof>，fill 保存 rhs（原样，含其前导空格；不含注释，因为注释留在 sketch 行内）
        if re.match(r'^\s*(have|suffices)\b', code) and (":=" in code) and (not re.search(r":=\s*by\b", code)):
            pos = code.index(":=")
            prefix = code[:pos + 2]     # 含 ':='
            rhs = code[pos + 2:]        # ':=' 右侧原样（含空格）
            fill.append(rhs)            # rhs 不含注释
            out.append(prefix + marker + comment)
            i += 1
            continue

        # ---------- (B) 行尾 `... by`（可能是 `:= by` / `by`） + 缩进块 ----------
        # 这里必须把“同一行的注释”也收入 fill，否则回填时注释会被挤到后续行
        m_end_by = re.search(r'\bby\s*$', code)
        if m_end_by:
            by_start = m_end_by.start()   # 从 'by' 的 b 开始替换
            prefix = code[:by_start]

            # 收集同一行从 'by' 开始到行尾（含注释）
            first_removed = code[by_start:] + comment

            # 收集后续属于该 by-block 的行：缩进 > base 或空行（与你原逻辑一致）
            j = i + 1
            removed_lines: List[str] = []
            while j < n and (_indent(raw_lines[j]) > base or raw_lines[j].strip() == ""):
                removed_lines.append(raw_lines[j])
                j += 1

            removed = first_removed
            if removed_lines:
                removed = removed + "\n" + "\n".join(removed_lines)

            fill.append(removed)
            # 注意：注释已放入 removed，所以 sketch 行不再拼 comment
            out.append(prefix + marker)
            i = j
            continue

        # ---------- (C) 行内 `by ...`（不跨行） ----------
        # 只在 code 部分做替换，comment 原样保留在 sketch 行
        def _repl(mm: re.Match) -> str:
            fill.append(mm.group(0))  # “by ...” 整段原样保存（含 by）
            return marker

        new_code = _BY_INLINE_RE.sub(_repl, code)
        out.append(new_code + comment)
        i += 1

    return "\n".join(out), fill



#### 检测函数

In [ ]:
# 检测将fill中内容回填sketch之后是否和原proof一致的函数


import difflib
from typing import List, Tuple, Optional, Literal

NormalizeMode = Literal["none", "rstrip", "strip"]

def refill_sketch(sketch: str, fills: List[str], hole_token: str = "<proof>") -> str:
    """
    将 fills 按出现顺序依次填回 sketch 的 <proof> 洞位，返回回填后的文本。
    要求：洞位数量 == len(fills)，否则抛出 ValueError。
    """
    if sketch is None:
        sketch = ""
    parts = sketch.split(hole_token)
    n_holes = len(parts) - 1
    if n_holes != len(fills):
        raise ValueError(f"洞位数量({n_holes}) != fills数量({len(fills)})")
    out = []
    for i in range(n_holes):
        out.append(parts[i])
        out.append(fills[i])
    out.append(parts[-1])
    return "".join(out)

def _canon(s: str, normalize: NormalizeMode) -> str:
    """统一换行符，并按 normalize 规则做轻量归一化。"""
    if s is None:
        s = ""
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    if normalize == "none":
        return s
    if normalize == "rstrip":
        # 每行去掉行尾空白；保留行首缩进（对 Lean 很重要）
        return "\n".join(line.rstrip() for line in s.split("\n"))
    if normalize == "strip":
        # 更宽松：每行行尾空白 + 整体首尾空白
        return "\n".join(line.rstrip() for line in s.split("\n")).strip()
    raise ValueError(f"unknown normalize={normalize}")

def check_refill_matches(
    proof: str,
    sketch: str,
    fills: List[str],
    hole_token: str = "<proof>",
    normalize: NormalizeMode = "none",
    show_diff: bool = True,
    context_lines: int = 3,
) -> Tuple[bool, str, Optional[str]]:
    """
    判断：refill_sketch(sketch, fills) 是否与 proof 一致。
    
    返回 (ok, filled_text, diff_str_or_None)
      - ok: bool
      - filled_text: 回填后的文本
      - diff_str: 若不一致且 show_diff=True，给出 unified diff；否则 None
    
    normalize:
      - "none"  : 严格逐字符比较（仅统一 CRLF->LF）
      - "rstrip": 推荐默认（忽略每行行尾空白差异）
      - "strip" : 更宽松（再忽略整体首尾空白）
    """
    filled = refill_sketch(sketch, fills, hole_token=hole_token)
    a = _canon(proof, normalize)
    b = _canon(filled, normalize)

    ok = (a == b)
    if ok or not show_diff:
        return ok, filled, None

    diff = "\n".join(
        difflib.unified_diff(
            a.split("\n"), b.split("\n"),
            fromfile="proof(original)",
            tofile="proof(refilled)",
            lineterm="",
            n=context_lines,
        )
    )
    return ok, filled, diff

# ok（bool）
#   True：把 fills 按顺序回填进 sketch 后，得到的文本 与原 proof 一致（会受 normalize 规则影响）。
#   False：不一致。

# filled（str）
#   回填后的完整 proof 文本（也就是 sketch 中每个 <proof> 都被对应的 fills[i] 替换后的结果）。
#   不管 ok True/False，都会返回这个字符串，方便你打印/保存/进一步检查。

# diff（Optional[str]）
#   如果 ok=False 且 show_diff=True：这里会给一个 unified diff（类似 git diff 的格式），显示“原 proof”和“回填后的 proof”哪里不一样。
#   如果 ok=True 或 show_diff=False：这里就是 None。

#### 检测结果

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path


with jsonlines.open(data_path, "r") as reader:
    
    Diff = []
    indent_mix = []
    for i, da in enumerate(reader, start=1):
        name = da.get("name")
        theorem = da.get("theorem")
        proof = da.get("proof")
        sketch, fill = sketchify_with_fills(proof)
        lean_block = da.get("lean_block")
        
#         print("\n name: \n", name)
      
#         print("\n theorem: \n", theorem)
        
#         print("\n proof:\n",proof)
       
#         print("\n sketch:\n",sketch)
#         print("\n fill:\n",fill)
#         print("\n")
        

        ok, filled, diff = check_refill_matches(proof, sketch, fill, normalize="none")
        if ok == False:
            Diff.append(name)
            
            
        make_proof = theorem + proof
        if make_proof != lean_block:
            print("\n make_proof: \n", make_proof)
            indent_mix.append(name)
            
        # print(ok)

print(Diff)          # 回填后与原 proof 不一致的数据 name 的 list
print(indent_mix)    # 将 theorem 和 proof 拼接后 index 出错的数据 name 的 list

## sketch.jsonl
#### 将data.jsonl中的proof生成sketch和fills，并保存

In [ ]:
from pathlib import Path
import jsonlines

def add_sketch_and_fill_to_jsonl(
    in_jsonl: str,
    out_jsonl: str | None = None,
    proof_key: str = "proof",
    sketch_key: str = "sketch",
    fill_key: str = "fills",
    overwrite: bool = False,
) -> str:
    """
    输入：in_jsonl (.jsonl)
    输出：out_jsonl (.jsonl)，在每条样本中新增 sketch / fills 字段，其它字段不变

    依赖：你已实现 sketchify_with_fills(proof_str) -> (sketch_str, fill_list)

    报错：
    - 文件中没有proof项：任意一条记录缺少 proof_key（或文件为空）
    - 文件中proof字段为空：任意一条记录 proof 为 None / 空串 / 仅空白
    
    overwrite：
    控制当输入 JSONL 里已经存在 sketch 或 fills 字段时，要不要覆盖它们。
    """
    in_path = Path(in_jsonl)
    if out_jsonl is None:
        out_path = in_path.with_suffix("")  # 去掉 .jsonl
        out_path = Path(str(out_path) + ".with_sketch.jsonl")
    else:
        out_path = Path(out_jsonl)

    if not in_path.exists():
        raise FileNotFoundError(f"找不到输入文件: {in_path}")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    seen_any = False

    with jsonlines.open(in_path, "r") as reader, jsonlines.open(out_path, "w") as writer:
        for obj in reader:
            seen_any = True

            # 1) proof 字段存在性
            if proof_key not in obj:
                raise KeyError("文件中没有proof项")

            proof_val = obj.get(proof_key)

            # 2) proof 字段非空
            if proof_val is None:
                raise ValueError("文件中proof字段为空")
            if not isinstance(proof_val, str):
                # 如果你希望允许非 str，可在这里改成 str(proof_val)
                raise TypeError(f"proof 字段必须是字符串，但得到了: {type(proof_val)}")
            if proof_val.strip() == "":
                raise ValueError("文件中proof字段为空")

            # 3) 避免覆盖已有字段（除非 overwrite=True）
            if (not overwrite) and (sketch_key in obj or fill_key in obj):
                raise ValueError(f"记录中已存在 {sketch_key}/{fill_key} 字段，拒绝覆盖（overwrite=False）")

            # 4) 生成 sketch / fill
            sketch, fills = sketchify_with_fills(proof_val)

            if not isinstance(sketch, str):
                raise TypeError("sketchify_with_fills 返回的 sketch 不是 str")
            if not isinstance(fills, list) or any(not isinstance(x, str) for x in fills):
                raise TypeError("sketchify_with_fills 返回的 fill 不是 list[str]")

            # 5) 保持其它字段不变，仅新增/写入 sketch/fill
            new_obj = dict(obj)
            new_obj[sketch_key] = sketch
            new_obj[fill_key] = fills

            writer.write(new_obj)

    if not seen_any:
        # 空文件：按“没有 proof 项”处理
        raise KeyError("文件中没有proof项")

    return str(out_path)


In [ ]:

sketch_path =  work_dir + "/sketch.jsonl"


In [ ]:
add_sketch_and_fill_to_jsonl(data_path, sketch_path)

#### 结果检测

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path


with jsonlines.open(sketch_path, "r") as reader:
    
    Diff = []
    
    indent_mix = []
    
    count = 0
    
    same = 0
    same_list = []
    
    for i, da in enumerate(reader, start=1):
        name = da.get("name")
        theorem = da.get("theorem")
        proof = da.get("proof")
        
        sketch = da.get("sketch")
        fills = da.get("fills")

        lean_block = da.get("lean_block")
        
        
#         print("\n name: \n", name)
      
#         print("\n theorem: \n", theorem)
        
#         print("\n proof:\n",proof)
       
#         print("\n sketch:\n",sketch)
#         print("\n fills:\n",fills)
#         print("\n")

        count += 1

        ok, filled, diff = check_refill_matches(proof, sketch, fills, normalize="none")
        if ok == False:
            Diff.append(name)
            
            
        make_proof = theorem + proof
        if make_proof != lean_block:
            print("\n make_proof: \n", make_proof)
            indent_mix.append(name)
            
        
        if sketch == proof:
            same += 1
            same_list.append(name)
        
      
        
print(Diff)
print(indent_mix)
print(count)
print(same)     # 无法 sketch 化的数据的数量
print(len(same_list))

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path


with jsonlines.open(sketch_path, "r") as reader:
    
    Diff = []
    indent_mix = []
    for i, da in enumerate(reader, start=1):
        name = da.get("name")
        theorem = da.get("theorem")
        proof = da.get("proof")
        
        sketch = da.get("sketch")
        fills = da.get("fills")

        lean_block = da.get("lean_block")
        
        
        print("\n name: \n", name)
      
        print("\n theorem: \n", theorem)
        
        print("\n proof:\n",proof)
       
        print("\n sketch:\n",sketch)
        print("\n fills:\n",fills)
        print("\n")
        

        ok, filled, diff = check_refill_matches(proof, sketch, fills, normalize="none")
        if ok == False:
            Diff.append(name)
            
            
        make_proof = theorem + proof
        if make_proof != lean_block:
            print("\n make_proof: \n", make_proof)
            indent_mix.append(name)
            
        # print(ok)
        
        
        if i >= 2:
            break

print(Diff)
print(indent_mix)

## sketch_only.jsonl

In [ ]:
from pathlib import Path
import jsonlines

def extract_real_sketch(
    in_jsonl: str,
    out_jsonl: str | None = None,
    proof_key: str = "proof",
    sketch_key: str = "sketch",
    normalize_newlines_for_compare: bool = True,
) -> tuple[str, int]:
    """
    作用：
      检测每条记录的 sketch 与 proof 是否完全一致；
      将所有“不一致”的记录（即 sketch 是真 sketch，而不是和 proof 一模一样）抽出保存。

    关键保证：
      - 输出时对每条记录“所有字段内容不做任何改动”（包括 proof/sketch 的缩进/空白信息）
      - 只做比较，不改写字段；写出时直接写原 obj

    参数：
      normalize_newlines_for_compare:
        True  : 比较时仅统一换行符（\r\n/\r -> \n），避免因为行尾格式导致误判
        False : 严格按原字符串逐字符比较（包括 \r\n 差异）

    返回：
      (输出文件路径, 写出的条目数)

    报错：
      - 任何一条缺少 proof 或 sketch 字段 -> KeyError
      - proof/sketch 不是字符串 -> TypeError
    """
    in_path = Path(in_jsonl)
    if not in_path.exists():
        raise FileNotFoundError(f"找不到输入文件: {in_path}")

    if out_jsonl is None:
        out_path = Path(str(in_path.with_suffix("")) + ".real_sketch.jsonl")
    else:
        out_path = Path(out_jsonl)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    def _norm(s: str) -> str:
        # 只用于“比较”，不用于写回
        return s.replace("\r\n", "\n").replace("\r", "\n")

    written = 0
    seen_any = False

    with jsonlines.open(in_path, "r") as reader, jsonlines.open(out_path, "w") as writer:
        for obj in reader:
            seen_any = True

            if proof_key not in obj:
                raise KeyError(f"文件中没有{proof_key}项")
            if sketch_key not in obj:
                raise KeyError(f"文件中没有{sketch_key}项")

            proof = obj[proof_key]
            sketch = obj[sketch_key]

            if not isinstance(proof, str):
                raise TypeError(f"{proof_key} 字段必须是 str，但得到 {type(proof)}")
            if not isinstance(sketch, str):
                raise TypeError(f"{sketch_key} 字段必须是 str，但得到 {type(sketch)}")

            a = _norm(proof) if normalize_newlines_for_compare else proof
            b = _norm(sketch) if normalize_newlines_for_compare else sketch

            # “真 sketch”判定：不完全一致
            if a != b:
                writer.write(obj)   # 原封不动写出（字段内容不变）
                written += 1

    if not seen_any:
        # 空文件通常也属于输入错误
        raise ValueError("输入文件为空，无法抽取")

    return str(out_path), written


In [ ]:

real_sketch_path = work_dir + "/real_sketch.jsonl"

In [ ]:
extract_real_sketch(sketch_path, real_sketch_path)

### 结果检测

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path


with jsonlines.open(real_sketch_path, "r") as reader:
    
    Diff = []
    
    indent_mix = []
    
    count = 0
    
    same = 0
    same_list = []
    
    for i, da in enumerate(reader, start=1):
        name = da.get("name")
        theorem = da.get("theorem")
        proof = da.get("proof")
        
        sketch = da.get("sketch")
        fills = da.get("fills")

        lean_block = da.get("lean_block")
        
        
#         print("\n name: \n", name)
      
#         print("\n theorem: \n", theorem)
        
#         print("\n proof:\n",proof)
       
#         print("\n sketch:\n",sketch)
#         print("\n fills:\n",fills)
#         print("\n")

        count += 1

        ok, filled, diff = check_refill_matches(proof, sketch, fills, normalize="none")
        if ok == False:
            Diff.append(name)
            
            
        make_proof = theorem + proof
        if make_proof != lean_block:
            print("\n make_proof: \n", make_proof)
            indent_mix.append(name)
            
        
        if sketch == proof:
            same += 1
            same_list.append(name)
        
      
        
print(Diff)
print(indent_mix)
print(count)
print(same)
print(len(same_list))